In [1]:
import numpy as np
import pandas as pd


In [2]:
#load GTF
gtffile= pd.read_csv('input/annotations_cleaned.gtf', index_col=None, skiprows=5, header=None, low_memory=False, delimiter="\t")
gtffile.columns=("chr","database","type","start","stop","unkown","strand","CDS","details")
gtffile["details"].str.split(';', n=0, expand=True)
genes=gtffile[gtffile["type"]=="gene"].copy()

step1=genes["details"].str.split(';', n=0, expand=True)
step2=step1[2].str.split('"', n=0, expand=True)
genes["genename"]=step2[1]
step3=step1[0].str.split('"', n=0, expand=True)
genes["geneID"]=step3[1]
step5=step1[4].str.split('"', n=0, expand=True)
genes["biotype"]=step5[1]
genes=genes[genes['genename']!='ensembl']
genes=genes[genes['biotype']=='protein_coding']

#remove gene duplicates and split by strand
genes=genes[np.invert(genes['genename'].duplicated(keep ='first'))]
genes[:5]

,chr,database,type,start,stop,unkown,strand,CDS,details,genename,geneID,biotype
6,1,ensembl_havana,gene,3205901,3671498,.,-,.,"gene_id ""ENSMUSG00000051951""; gene_version ""5""...",Xkr4,ENSMUSG00000051951,protein_coding
70,1,ensembl_havana,gene,3999557,4409241,.,-,.,"gene_id ""ENSMUSG00000025900""; gene_version ""12...",Rp1,ENSMUSG00000025900,protein_coding
171,1,ensembl_havana,gene,4490931,4497354,.,-,.,"gene_id ""ENSMUSG00000025902""; gene_version ""13...",Sox17,ENSMUSG00000025902,protein_coding
294,1,ensembl_havana,gene,4773206,4785739,.,-,.,"gene_id ""ENSMUSG00000033845""; gene_version ""13...",Mrpl15,ENSMUSG00000033845,protein_coding
365,1,ensembl_havana,gene,4807788,4848410,.,+,.,"gene_id ""ENSMUSG00000025903""; gene_version ""14...",Lypla1,ENSMUSG00000025903,protein_coding


In [3]:
#define windows based on selected extension relative to the TSS
distup=2500
distdown=2500

plus=genes[genes["strand"]=="+"].copy()
minus=genes[genes["strand"]=="-"].copy()

plusstart=plus["start"]-distup
plusstop=plus["start"]+distdown
plusfinal=pd.DataFrame({"chr":plus["chr"],"start":plusstart,"stop":plusstop,"name":plus["genename"]})
minusstart=minus["stop"]-distdown
minusstop=minus["stop"]+distup
minusfinal=pd.DataFrame({"chr":minus["chr"],"start":minusstart,"stop":minusstop,"name":minus["genename"]})
final=pd.concat([plusfinal,minusfinal])

#Shorten windows in cases where they overlap the end of the chromsomes
final.loc[final['start']<0,'start']=0
final.loc[final['stop']<0,'stop']=0

final.to_csv('output/TSS_5kb_genes.bed',header=None, index=None, sep="\t")

In [4]:
#define windows based on selected extension relative to the TSS
distup=0
distdown=10000

plus=genes[genes["strand"]=="+"].copy()
minus=genes[genes["strand"]=="-"].copy()

plusstart=plus["start"]-distup
plusstop=plus["start"]+distdown
plusfinal=pd.DataFrame({"chr":plus["chr"],"start":plusstart,"stop":plusstop,"name":plus["genename"]})
minusstart=minus["stop"]-distdown
minusstop=minus["stop"]+distup
minusfinal=pd.DataFrame({"chr":minus["chr"],"start":minusstart,"stop":minusstop,"name":minus["genename"]})
final=pd.concat([plusfinal,minusfinal])

#Shorten windows in cases where they overlap the end of the chromsomes
final.loc[final['start']<0,'start']=0
final.loc[final['stop']<0,'stop']=0

final.to_csv('output/TSS_10kb_inside_genes.bed',header=None, index=None, sep="\t")